# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 04

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. После сортировки таблицы по балансу `df_sorted = df.sort_values(by='balance', ascending=False)` аналитик заметил, что числа в крайнем левом столбце идут вразнобой (например, 14, 2, 88...).
Что представляет собой этот столбец и какой метод позволяет восстановить непрерывную нумерацию строк от 0?

**Ответ:**
Этот столбец — **индекс строк** (`df.index`), сохранивший исходные номера записей до сортировки. Чтобы восстановить последовательную нумерацию от 0, вызывается метод `df_sorted.reset_index(drop=True)`.

---

#### 1.2. Для чего в анализе данных используется атрибут `df.shape`?
Что вернет этот атрибут для таблицы из 150 клиентов и 7 столбцов, и под каким индексом кортежа (`[0]` или `[1]`) хранится количество строк?

**Ответ:**
`df.shape` возвращает размерность таблицы в виде кортежа `(150, 7)`. Количество строк хранится под индексом `[0]`, а количество столбцов — под индексом `[1]`.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["discounted_total"] = (df["total_paid"] * 0.9).round(2)
```

**Ответ:**
Рассчитывает сумму к оплате с учетом дополнительной скидки 10% (умножение на 0.9) и округляет итоговую сумму до 2 знаков после запятой.

---

#### 1.4. В чем ключевая разница между фильтрацией и агрегацией строк датафрейма?
```python
# Действие А:
df[df['total_paid'] > 500]
# Действие Б:
df.groupby('game_title')['total_paid'].sum()
```
Какое действие отбирает существующие строки без изменения их содержимого, а какое — сжимает (схлопывает) множество строк в одну?

**Ответ:**
* Действие А выполняет **фильтрацию**: оно отбирает исходные строки сессий с оплатой > 500 руб., сохраняя структуру каждой строки.
* Действие Б выполняет **агрегацию**: оно объединяет строки в группы по играм и сворачивает их в одну итоговую сумму на каждую игру.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `game_sessions.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path("../variant_04")
df_audit = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Средняя длительность сессии: 2.23")

Количество строк в таблице: 500
Столбцы датафрейма: ['session_id', 'member_id', 'pc_id', 'game_title', 'session_date', 'duration_hours', 'discount_promo', 'total_paid', 'client_rating']
Средняя длительность сессии: 2.23

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `game_sessions.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path(f"../variant_04")
df_data = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

summary = df_data.groupby("game_title").agg({
    "total_paid": "sum",
    "duration_hours": "mean"
}).reset_index()

summary["duration_hours"] = summary["duration_hours"].round(2)
summary = summary.sort_values(by="total_paid", ascending=False).reset_index(drop=True)
display(summary)

,game_title,total_paid,duration_hours
0,PUBG,31844,2.32
1,Atomic Heart,29132,2.33
2,Rust,25599,2.29
3,Apex Legends,23412,2.29
4,Valorant,22306,2.17
5,GTA V,22066,2.20
6,Dota 2,21804,2.24
7,Warzone,21733,2.06
8,Cyberpunk 2077,21305,2.18
9,Counter-Strike 2,19393,2.14


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `game_sessions` для ряда сессий значение в столбце `client_rating` равно `NULL`.
Чем для СУБД и для аналитика значение `NULL` принципиально отличается от оценки `0` при расчете среднего рейтинга `AVG(client_rating)`?

**Ответ:**
`NULL` означает отсутствие значения (клиент не оставил оценку), а `0` — это конкретная выставленная оценка. При вычислении `AVG(client_rating)` СУБД автоматически игнорирует строки со значением `NULL`, тогда как оценка `0` вошла бы в расчет и сильно занизила средний рейтинг клуба.

---

#### 3.2. Администратор киберклуба случайно выполнил запрос `DELETE FROM game_sessions;` вместо удаления одной устаревшей сессии.
Если запрос выполнялся внутри открытой транзакции, какой командой можно отменить удаление до того, как изменения зафиксируются на диске?

**Ответ:**
Командой `ROLLBACK`. Она отменяет все операции текущей открытой транзакции и возвращает данные в исходное состояние (до вызова команды `COMMIT`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT city, COUNT(*) AS members_cnt, AVG(balance) AS avg_bal FROM members GROUP BY city ORDER BY members_cnt DESC;
```

**Ответ:**
Запрос группирует клиентов по городам (`city`) и вычисляет число клиентов (`COUNT(*)`) и их средний баланс (`AVG(balance)`), сортируя по убыванию числа клиентов.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_04")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT pc_id, pc_number, zone_name, hourly_rate, gpu, ram_gb, status
FROM hardware_pcs
WHERE status = 'Active' AND ram_gb >= 32
ORDER BY hourly_rate DESC, pc_number ASC;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,pc_id,pc_number,zone_name,hourly_rate,gpu,ram_gb,status
0,24,PC-24,VIP,400.0,RTX 4070 Ti,64,Active
1,25,PC-25,VIP,400.0,RTX 4080 Super,64,Active
2,26,PC-26,VIP,400.0,RTX 4080 Super,64,Active
3,23,PC-23,VIP,380.0,RTX 4090,64,Active
4,15,PC-15,Bootcamp,270.0,RTX 3060 Ti,32,Active
5,18,PC-18,Bootcamp,270.0,RTX 4060 Ti,32,Active
6,20,PC-20,Bootcamp,270.0,RTX 4060,32,Active
7,16,PC-16,Bootcamp,250.0,RTX 3060 Ti,32,Active
8,17,PC-17,Bootcamp,250.0,RTX 3060 Ti,32,Active
9,19,PC-19,Bootcamp,250.0,RTX 4060,32,Active


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT zone_name, COUNT(*) AS pcs_count, ROUND(AVG(hourly_rate), 2) AS avg_rate, MAX(hourly_rate) AS max_rate
FROM hardware_pcs
GROUP BY zone_name
HAVING AVG(hourly_rate) > 200
ORDER BY avg_rate DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,zone_name,pcs_count,avg_rate,max_rate
0,VIP,5,400.00,420.0
1,PS5,3,306.67,320.0
2,Bootcamp,8,255.00,270.0


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT m.member_id, m.nickname, m.loyalty_tier, COUNT(s.session_id) AS sessions_count, ROUND(SUM(s.duration_hours), 2) AS total_hours
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
WHERE m.city = 'Москва'
GROUP BY m.member_id, m.nickname, m.loyalty_tier
ORDER BY total_hours DESC
LIMIT 8;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,loyalty_tier,sessions_count,total_hours
0,38,Kraken_37,Gold,11,31.0
1,30,Echo_889,Gold,9,25.0
2,15,Bullet_991,Bronze,9,18.5
3,78,Wolf_912,Platinum,6,17.0
4,35,Phoenix_32,Platinum,4,16.0
5,37,Shade_935,Gold,6,15.5
6,88,Warlock_652,Bronze,5,15.5
7,70,Apex_179,Bronze,8,14.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT pc_id, pc_number, zone_name, hourly_rate, ram_gb
FROM hardware_pcs
WHERE hourly_rate > (SELECT AVG(hourly_rate) FROM hardware_pcs)
ORDER BY hourly_rate DESC;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,pc_id,pc_number,zone_name,hourly_rate,ram_gb
0,27,PC-27,VIP,420.0,64
1,24,PC-24,VIP,400.0,64
2,25,PC-25,VIP,400.0,64
3,26,PC-26,VIP,400.0,64
4,23,PC-23,VIP,380.0,64
5,29,PC-29,PS5,320.0,16
6,28,PC-28,PS5,300.0,16
7,30,PC-30,PS5,300.0,16
8,15,PC-15,Bootcamp,270.0,32
9,18,PC-18,Bootcamp,270.0,32


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Список клиентов премиального уровня

**Найденные ошибки:**
1. Ошибка строкового литерала: текстовое значение `Platinum` указано без кавычек, из-за чего СУБД воспринимает его как имя несуществующего столбца.

In [9]:
query_51_fixed = '''
SELECT 
    member_id,
    nickname,
    city,
    loyalty_tier,
    balance
FROM members
WHERE loyalty_tier = 'Platinum'
ORDER BY balance DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,member_id,nickname,city,loyalty_tier,balance
0,27,Mystic_540,Москва,Platinum,4000.0
1,13,Hunter_972,Казань,Platinum,2500.0
2,125,Striker_234,Санкт-Петербург,Platinum,1500.0
3,131,Warlock_356,Москва,Platinum,1500.0
4,78,Wolf_912,Москва,Platinum,1000.0
5,81,Frost_274,Санкт-Петербург,Platinum,500.0
6,110,Blaze_341,Москва,Platinum,300.0
7,35,Phoenix_32,Москва,Platinum,150.0
8,96,Storm_361,Екатеринбург,Platinum,150.0


#### 5.2. Выручка по игровым местам

**Найденные ошибки:**
1. Ошибка ключа соединения: `p.pc_id = s.member_id` связывает компьютер с идентификатором клиента вместо `s.pc_id`.
2. Агрегатная функция в `WHERE`: условие `SUM(s.total_paid) > 5000` должно быть перенесено в блок `HAVING`.

In [10]:
query_52_fixed = '''
SELECT 
    p.pc_id,
    p.pc_number,
    p.zone_name,
    SUM(s.total_paid) AS pc_revenue
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
GROUP BY p.pc_id, p.pc_number, p.zone_name
HAVING SUM(s.total_paid) > 5000
ORDER BY pc_revenue DESC
LIMIT 10;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,pc_id,pc_number,zone_name,pc_revenue
0,23,PC-23,VIP,23959.0
1,17,PC-17,Bootcamp,15826.0
2,25,PC-25,VIP,15680.0
3,18,PC-18,Bootcamp,14323.0
4,22,PC-22,Bootcamp,11040.0
5,26,PC-26,VIP,11040.0
6,15,PC-15,Bootcamp,10219.0
7,16,PC-16,Bootcamp,9975.0
8,10,PC-10,Standard,9801.0
9,8,PC-08,Standard,9697.0


#### 5.3. Анализ активности и расходов клиентов

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = s.pc_id` связывает идентификатор клиента с номером компьютера вместо `s.member_id`.
2. Агрегатная функция в `WHERE`: условие `WHERE SUM(s.total_paid) > 1200` должно быть перенесено в блок `HAVING`.
3. Неполный `GROUP BY`: столбцы `m.nickname` и `m.city` присутствуют в `SELECT`, но пропущены в списке группировки.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    COUNT(s.session_id) AS cnt,
    SUM(s.total_paid) AS total_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city
HAVING SUM(s.total_paid) > 1200
ORDER BY total_spent DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,city,cnt,total_spent
0,38,Kraken_37,Москва,11,7290.0
1,30,Echo_889,Москва,9,5770.0
2,24,Bullet_938,Новосибирск,8,5385.0
3,44,Reaper_818,Санкт-Петербург,8,5249.0
4,15,Bullet_991,Москва,9,5193.0
5,107,Blaze_701,Санкт-Петербург,7,4275.0
6,78,Wolf_912,Москва,6,4200.0
7,74,Matrix_127,Санкт-Петербург,8,4090.0
8,88,Warlock_652,Москва,5,4057.0
9,115,Viper_528,Санкт-Петербург,7,3865.0
